<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎤 Dots.TTS — Zero-Shot Voice Cloning & TTS</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Colab Free Tier (T4 GPU) — Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>2B Parameter Continuous Autoregressive Text-to-Speech system</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Features
1. **Zero-Shot Voice Cloning**: Clone any voice using a short reference audio clip (3–10s) and synthesize speech with high naturalness.
2. **Multi-Model Options**:
   - **`dots.tts-base`**: Pretrained foundation model, perfect for general voice generation.
   - **`dots.tts-soar`**: Refined via Self-Corrective Alignment (SCA) for highest speaker similarity.
   - **`dots.tts-mf`**: MeanFlow distilled variant, optimized for low-latency, ultra-fast inference (NFE=4).
3. **Automatic Transcription**: If the reference audio transcript is left blank, a lightweight Whisper model will automatically transcribe it for you.
4. **T4 Memory & Speed Optimizations**: Auto-cleans GPU cache and maps precision settings (`float16` vs `bfloat16`) to fit Tesla T4 VRAM.

---

### Quick Start
1. Ensure **GPU runtime** is active: **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order.
3. Access the public Gradio link generated at the end to clone voices!


In [ ]:
#@title 🖥️ Step 1: Environment Setup
# Drops cache files and optimizes GPU memory allocations for Colab's RAM limits.
import os, gc, psutil

print('=== Colab T4 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')

In [ ]:
#@title 📦 Step 2: Install System Dependencies & Python Libraries
# Installs ffmpeg and the dots.tts package while keeping Colab's pre-installed NumPy 2.x version.
import subprocess
import torch
import os
import sys

try:
    subprocess.run(['nvidia-smi'], check=True)
    print(f'GPU Active: {torch.cuda.get_device_name(0)}')
except Exception:
    print('WARNING: No GPU active. Go to Runtime → Change runtime type → T4 GPU')

print('Installing ffmpeg...')
subprocess.run(['apt-get', 'update'], stdout=subprocess.DEVNULL)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], stdout=subprocess.DEVNULL)

# 1. Clone dots.tts locally
if not os.path.exists('dots.tts'):
    print('Cloning dots.tts repository...')
    subprocess.run(['git', 'clone', 'https://github.com/rednote-hilab/dots.tts.git'], check=True)
else:
    print('dots.tts repository already cloned.')

# 2. Patch pyproject.toml to remove the numpy>=2.2.6 constraint.
# This prevents pip from upgrading/downgrading NumPy, preserving Colab's pre-installed NumPy 2.x.
toml_path = 'dots.tts/pyproject.toml'
if os.path.exists(toml_path):
    with open(toml_path, 'r') as f:
        toml_content = f.read()

    # Replace numpy>= constraints with numpy to preserve Colab's default NumPy 2.x
    import re
    patched_content = re.sub(r'["\']numpy>=[0-9\.]+["\']', '"numpy"', toml_content)
    patched_content = re.sub(r'numpy>=[0-9\.]+', 'numpy', patched_content)

    with open(toml_path, 'w') as f:
        f.write(patched_content)
    print("✅ Patched pyproject.toml to preserve pre-installed NumPy 2.x version.")

# 3. Fetch and optimize constraints file (ignore PyTorch and NumPy)
url = "https://raw.githubusercontent.com/rednote-hilab/dots.tts/main/constraints/recommended.txt"
local_constraints = "constraints.txt"

try:
    print("Downloading and optimizing package installation constraints...")
    import urllib.request
    with urllib.request.urlopen(url) as response:
        content = response.read().decode('utf-8')

    lines = content.splitlines()
    filtered_lines = []
    for line in lines:
        line_strip = line.strip()
        if not line_strip or line_strip.startswith('#'):
            filtered_lines.append(line)
            continue
        match = re.match(r'^([a-zA-Z0-9_\-]+)', line_strip)
        if match:
            pkg_name = match.group(1).lower()
            if pkg_name in ['torch', 'torchaudio', 'numpy', 'numba']:
                print(f"  - Preserving pre-installed: '{pkg_name}'")
                continue
        filtered_lines.append(line)

    with open(local_constraints, 'w') as f:
        f.write('\n'.join(filtered_lines))
    print("✅ Constraints optimized successfully.")
except Exception as e:
    print(f"Warning: Constraints optimization failed ({e}), falling back to direct install.")
    local_constraints = None

# 4. Live pip installation helper
def run_pip_install(packages, constraints=None):
    import sys
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install']
    if constraints:
        cmd += ['-c', constraints]
    cmd += packages
    print(f"Running: {' '.join(cmd)}")
    sys.stdout.flush()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
            sys.stdout.flush()
    rc = process.poll()
    if rc != 0:
        raise RuntimeError(f"pip install failed with return code {rc}")

print('Installing dots.tts and requirements locally...')
run_pip_install(
    packages=['./dots.tts'],
    constraints=local_constraints
)

print('Installing Gradio and additional tools...')
run_pip_install(
    packages=['gradio', 'soundfile', 'transformers']
)

print('✅ Dependencies installed successfully!')

# 5. Sanity check imports right inside Step 2
try:
    print("Running sanity check on imports...")
    import sys
    for p in ['/content/dots.tts', os.path.abspath('dots.tts')]:
        if os.path.exists(p) and p not in sys.path:
            sys.path.insert(0, p)
    import numpy as np
    print(f"Active NumPy version: {np.__version__}")
    from transformers import pipeline
    from dots_tts.runtime import DotsTtsRuntime
    print("✅ All imports validated! Ready to run Step 3.")
except Exception as e:
    print(f"⚠️ Import validation warning: {e}")
    print("\n" + "="*80)
    print("❗ ACTION REQUIRED: Please manually restart the session to clear memory:")
    print("   Go to the top menu: Runtime -> Restart session")
    print("   Once reconnected, run Step 3 directly.")
    print("="*80 + "\n")

In [ ]:
#@title 🚀 Step 3: Launch Gradio Web Interface
import sys
import os

# Set up paths for local dots.tts package import
for p in ['/content/dots.tts', os.path.abspath('dots.tts')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import gc
import tempfile
import traceback
import psutil
import torch
import soundfile as sf
import gradio as gr
from transformers import pipeline
from dots_tts.runtime import DotsTtsRuntime

# Setup directories
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {DEVICE}")

# ASR Model pipeline for auto-transcribing reference voice
ASR_PIPELINE = None

def get_asr_pipeline():
    global ASR_PIPELINE
    if ASR_PIPELINE is None:
        print("Loading Whisper-base for auto-transcription...")
        ASR_PIPELINE = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-base",
            device=DEVICE
        )
    return ASR_PIPELINE

# Cache for active model runtime
CURRENT_MODEL_PATH = None
CURRENT_PRECISION = None
RUNTIME = None

def get_runtime(model_path, precision):
    global CURRENT_MODEL_PATH, CURRENT_PRECISION, RUNTIME

    # Reload if model or precision changes
    if RUNTIME is not None and CURRENT_MODEL_PATH == model_path and CURRENT_PRECISION == precision:
        return RUNTIME

    if RUNTIME is not None:
        print(f"Releasing previous model {CURRENT_MODEL_PATH} ({CURRENT_PRECISION}) from VRAM...")
        del RUNTIME
        RUNTIME = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"Loading model '{model_path}' in {precision}...")
    CURRENT_MODEL_PATH = model_path
    CURRENT_PRECISION = precision

    RUNTIME = DotsTtsRuntime.from_pretrained(
        model_path,
        precision=precision,
        optimize=False  # Disabled torch.compile to eliminate runtime compilation overhead and make inference fast
    )
    print("Model loaded successfully!")
    return RUNTIME

# Inference handler
def synthesize(ref_audio, ref_text, gen_text, model_name, precision, num_steps, guidance_scale, speaker_scale, ode_method, speed_factor, progress=gr.Progress()):
    if DEVICE == "cpu":
        return None, "❌ Error: CPU runtime detected! Go to Runtime → Change runtime type and select T4 GPU."

    if ref_audio is None:
        return None, "⚠️ Please upload or record a Reference Audio first."

    if not gen_text.strip():
        return None, "⚠️ Please enter the text you want to synthesize."

    try:
        progress(0.1, desc="Loading/initializing TTS model...")
        runtime = get_runtime(model_name, precision)

        # Transcript handling
        ref_text_clean = ref_text.strip()
        if not ref_text_clean:
            progress(0.3, desc="ASR: Auto-transcribing reference audio...")
            asr = get_asr_pipeline()
            res = asr(ref_audio)
            ref_text_clean = res["text"].strip()
            print(f"Auto-transcribed transcript: '{ref_text_clean}'")

        # Punctuation guard for autoregressive synthesis
        gen_text_clean = gen_text.strip()
        if gen_text_clean and gen_text_clean[-1] not in '.!?,;。！？，；':
            gen_text_clean += '.'

        progress(0.5, desc="Synthesizing audio (Autoregressive Flow Matching)...")
        print(f"Synthesizing: '{gen_text_clean}'")
        print(f"Params: model={model_name}, precision={precision}, steps={num_steps}, cfg={guidance_scale}, speaker_scale={speaker_scale}, ode={ode_method}, speed={speed_factor}")

        # Run inference
        result = runtime.generate(
            text=gen_text_clean,
            prompt_audio_path=ref_audio,
            prompt_text=ref_text_clean,
            num_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            speaker_scale=float(speaker_scale),
            ode_method=ode_method
        )

        # 1. Extract audio if result is a dictionary
        if isinstance(result, dict):
            if "audio" in result:
                result = result["audio"]
            else:
                # Fallback to the first value that looks like audio data
                for val in result.values():
                    if hasattr(val, "detach") or hasattr(val, "ndim") or isinstance(val, (list, tuple)):
                        result = val
                        break

        # 2. Convert PyTorch tensor to CPU NumPy array
        if hasattr(result, "detach"):
            result = result.detach().cpu()
            if hasattr(result, "float"):
                result = result.float()
            result = result.numpy()

        # 3. Convert list/tuple to NumPy array
        if isinstance(result, (list, tuple)):
            import numpy as np
            result = np.array(result)

        # 4. Squeeze out extra dimensions to get 1D array
        if hasattr(result, "ndim") and result.ndim > 1:
            result = result.squeeze()

        # 5. Ensure it's a 1D NumPy float32 array
        import numpy as np
        if not isinstance(result, np.ndarray):
            result = np.array(result, dtype=np.float32)
        else:
            result = result.astype(np.float32)

        # 6. Apply speed adjustment if speed_factor is not 1.0
        if abs(float(speed_factor) - 1.0) > 0.01:
            progress(0.95, desc="Adjusting speech speed (time stretch)...")
            import librosa
            result = librosa.effects.time_stretch(result, rate=float(speed_factor))

        progress(0.98, desc="Writing output audio...")
        out_dir = tempfile.mkdtemp()
        out_path = os.path.join(out_dir, "output.wav")
        # dots.tts outputs mono audio at 48kHz sampling rate
        sf.write(out_path, result, 48000)

        progress(1.0, desc="Success!")
        status_msg = f"✅ Success!\nReference Transcript: {ref_text_clean}"
        return out_path, status_msg

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# Gradio Custom Styling
CSS = """@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@400;500;600;700&display=swap');
* { font-family: 'Outfit', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 32px; border-radius: 16px; margin-bottom: 24px; box-shadow: 0 10px 30px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2.2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1.1em; margin-bottom: 18px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 600; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
.footer { text-align: center; padding: 24px; margin-top: 36px; border-top: 1px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(title="Dots.TTS — Zero-Shot Voice Cloning | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎤 Dots.TTS — Zero-Shot Voice Cloning & TTS</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | Colab Free Tier</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        "Upload a reference voice sample (3 to 10 seconds), type the target text, "
        "and hit **Generate Audio**. If the transcript field is blank, the UI will auto-transcribe it."
    )

    with gr.Row():
        with gr.Column(scale=1):
            ref_audio = gr.Audio(label="📎 Reference Audio (3–10s)", type="filepath")
            ref_text = gr.Textbox(label="Reference Transcript (Optional)", placeholder="Leave blank to auto-transcribe...")
            gen_text = gr.Textbox(label="Text to Synthesize", placeholder="Enter text to generate...", lines=3)

            model_name = gr.Dropdown(
                label="Model Checkpoint",
                choices=["rednote-hilab/dots.tts-base", "rednote-hilab/dots.tts-soar", "rednote-hilab/dots.tts-mf"],
                value="rednote-hilab/dots.tts-mf"
            )

            precision = gr.Dropdown(
                label="Model Precision",
                choices=["float16", "bfloat16", "float32"],
                value="float16"
            )

        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Advanced Inference Parameters", open=True):
                num_steps = gr.Slider(label="⚡ Flow Matching Steps", minimum=2, maximum=50, step=1, value=4)
                guidance_scale = gr.Slider(label="🎯 Classifier Free Guidance (CFG)", minimum=0.5, maximum=2.0, step=0.1, value=1.0)
                speaker_scale = gr.Slider(label="👤 Speaker Intensity", minimum=0.5, maximum=2.5, step=0.1, value=1.5)
                ode_method = gr.Dropdown(label="🧩 ODE Solver", choices=["euler", "midpoint", "rk4"], value="euler")
                speed_factor = gr.Slider(label="🎚️ Speech Speed (Time Stretch)", minimum=0.5, maximum=2.0, step=0.1, value=1.0)

            with gr.Row():
                gen_btn = gr.Button("🎬 Generate Audio", variant="primary", size="lg")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

            audio_out = gr.Audio(label="🔊 Synthesized Audio Output", type="filepath")
            status_out = gr.Textbox(label="Status & Transcript Output", interactive=False)

    # Auto-update parameters based on model selected
    def update_model_params(model):
        if "dots.tts-mf" in model:
            # MeanFlow is distilled for 4 steps, guidance is no-op
            return gr.update(value=4), gr.update(value=1.0, interactive=False)
        else:
            return gr.update(value=10), gr.update(value=1.2, interactive=True)

    model_name.change(
        fn=update_model_params,
        inputs=[model_name],
        outputs=[num_steps, guidance_scale]
    )

    gen_event = gen_btn.click(
        fn=synthesize,
        inputs=[ref_audio, ref_text, gen_text, model_name, precision, num_steps, guidance_scale, speaker_scale, ode_method, speed_factor],
        outputs=[audio_out, status_out]
    )

    stop_btn.click(fn=None, cancels=[gen_event])

    clear_btn.click(
        fn=lambda: (None, "", "", "rednote-hilab/dots.tts-mf", "float16", 4, 1.0, 1.5, "euler", 1.0, None, ""),
        outputs=[ref_audio, ref_text, gen_text, model_name, precision, num_steps, guidance_scale, speaker_scale, ode_method, speed_factor, audio_out, status_out]
    )

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎙️ Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | Zero-Shot Voice Cloning | Colab Free Tier</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("Launching Gradio demo...")
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, css=CSS, theme=gr.themes.Soft())